# AO3 Loader Prep Notebook

This notebook is a working bridge between the scraped AO3 CSV and the PostgreSQL loader.

It does three things:

1. Loads the raw export from `data/ao3_data.csv`.
2. Profiles and cleans the messy columns.
3. Builds normalized tables that can later be moved into `load_ao3.py`.

In [1]:
from pathlib import Path
import ast
import re

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 140)

In [2]:
df_raw = pd.read_csv('../data/ao3_data.csv')
df_raw.head(3)

,title,author,summary,words,current_chapters,total_chapters,kudos,bookmarks,hits,rating,language,status,last_date,warning,category,fandom,relationship,character,freeform,url
0,The man I hate the most 😏.,/users/Pickle_leaf_22/pseuds/Pickle_leaf_22,Aizawa spends time thinking about someone he hates but deep inside he never really hated this so called person. Afte...,"1,754",1,1,NaN,NaN,32,Explicit,English,NaN,NaN,['Creator Chose Not To Use Archive Warnings'],['M/M'],['僕のヒーローアカデミア | Boku no Hero Academia | My Hero Academia (Anime & Manga)'],['Aizawa Shouta | Eraserhead/Yagi Toshinori | All Might'],"['Aizawa Shouta | Eraserhead', 'Yagi Toshinori | All Might']","['Warm and Fuzzy Feelings', 'Love Confessions', 'Old man yaoi', 'Porn with Feelings', 'Porn With Some Plot', 'Sex To...",https://archiveofourown.org/works/88298181?view_adult=true
1,Glitch,/users/blop_the_bloblin/pseuds/blop_the_bloblin,"Please just read the first chapter if you’re curious, it’s only like, 600 words, and I really don’t know what else t...",642,1,NaN,6,3,123,Teen And Up Audiences,English,Updated:,2026-07-10,['Creator Chose Not To Use Archive Warnings'],"['F/M', 'Other']",['僕のヒーローアカデミア | Boku no Hero Academia | My Hero Academia (Anime & Manga)'],"['Aizawa Shouta | Eraserhead & Midoriya Izuku', 'Midoriya Izuku/Yaoyorozu Momo']","['Midoriya Izuku', 'Yaoyorozu Momo', 'Bakugou Katsuki', 'Aizawa Shouta | Eraserhead', 'Midoriya Inko', 'Yagi Toshino...","['Original Character(s)', 'Midoriya Izuku Has a Quirk', 'Midoriya Izuku Does Not Have One for All Quirk', 'Slow Burn...",https://archiveofourown.org/works/88296531?view_adult=true
2,Keeping a Safe Distance,/users/Frol211/pseuds/Frol211,Hawks is sent undercover into the League of Villains. His mission: earn their trust.\r\nBut the safe distance betwee...,"2,857",2,13,2,NaN,44,Explicit,English,Updated:,2026-07-10,['Creator Chose Not To Use Archive Warnings'],['M/M'],['僕のヒーローアカデミア | Boku no Hero Academia | My Hero Academia (Anime & Manga)'],['Dabi | Todoroki Touya/Takami Keigo | Hawks'],"['Dabi | Todoroki Touya', 'Takami Keigo | Hawks', 'Shigaraki Tomura | Shimura Tenko', 'Kurogiri (My Hero Academia)',...","['Slow Burn', 'Enemies to Lovers', 'Mutual Pining', 'Undercover', 'Canon Divergence', 'Angst', 'Hurt/Comfort', 'Leag...",https://archiveofourown.org/works/88295646?view_adult=true


In [3]:
df_raw.shape, df_raw.columns.tolist()

((7040, 20),
 ['title',
  'author',
  'summary',
  'words',
  'current_chapters',
  'total_chapters',
  'kudos',
  'bookmarks',
  'hits',
  'rating',
  'language',
  'status',
  'last_date',
  'warning',
  'category',
  'fandom',
  'relationship',
  'character',
  'freeform',
  'url'])

## Raw data audit

This section checks nulls, sample values, and the messy string/list columns before any transformation.

In [4]:
df_raw.isna().sum().sort_values(ascending=False)

total_chapters      3660
status              2112
last_date           2112
bookmarks           1759
kudos                497
summary              170
author               105
title                  0
words                  0
current_chapters       0
rating                 0
hits                   0
language               0
warning                0
category               0
fandom                 0
relationship           0
character              0
freeform               0
url                    0
dtype: int64

In [5]:
list_columns = ['warning', 'category', 'fandom', 'relationship', 'character', 'freeform']
sample_lists = {column: df_raw[column].dropna().astype(str).head(3).tolist() for column in list_columns if column in df_raw.columns}
sample_lists

{'warning': ["['Creator Chose Not To Use Archive Warnings']",
  "['Creator Chose Not To Use Archive Warnings']",
  "['Creator Chose Not To Use Archive Warnings']"],
 'category': ["['M/M']", "['F/M', 'Other']", "['M/M']"],
 'fandom': ["['僕のヒーローアカデミア | Boku no Hero Academia | My Hero Academia (Anime & Manga)']",
  "['僕のヒーローアカデミア | Boku no Hero Academia | My Hero Academia (Anime & Manga)']",
  "['僕のヒーローアカデミア | Boku no Hero Academia | My Hero Academia (Anime & Manga)']"],
 'relationship': ["['Aizawa Shouta | Eraserhead/Yagi Toshinori | All Might']",
  "['Aizawa Shouta | Eraserhead & Midoriya Izuku', 'Midoriya Izuku/Yaoyorozu Momo']",
  "['Dabi | Todoroki Touya/Takami Keigo | Hawks']"],
 'character': ["['Aizawa Shouta | Eraserhead', 'Yagi Toshinori | All Might']",
  "['Midoriya Izuku', 'Yaoyorozu Momo', 'Bakugou Katsuki', 'Aizawa Shouta | Eraserhead', 'Midoriya Inko', 'Yagi Toshinori | All Might']",
  "['Dabi | Todoroki Touya', 'Takami Keigo | Hawks', 'Shigaraki Tomura | Shimura Tenko', 'Ku

In [6]:
integer_columns = ['words', 'kudos', 'bookmarks', 'hits', 'current_chapters', 'total_chapters']
df_raw[integer_columns].head(5)

,words,kudos,bookmarks,hits,current_chapters,total_chapters
0,"1,754",NaN,NaN,32,1,1
1,642,6,3,123,1,NaN
2,"2,857",2,NaN,44,2,13
3,"1,058",1,NaN,13,1,1
4,"1,158",NaN,1,20,1,NaN


## Cleaning helpers

These functions mirror the transformation logic the loader will need later.

In [7]:
def clean_int(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().replace(',', '')
    if text == '':
        return pd.NA
    return int(text)


def clean_status(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().lower()
    if text.startswith('completed'):
        return 'completed'
    if text.startswith('updated'):
        return 'on_going'
    return text or pd.NA


def clean_date(value):
    if pd.isna(value):
        return pd.NaT
    return pd.to_datetime(value, errors='coerce').date()


def clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = re.sub(r'\s+', ' ', str(value)).strip()
    return text if text else pd.NA


def parse_list_literal(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text in {'', '[]', 'nan', 'None'}:
        return []
    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return [item.strip() for item in re.split(r'\s*,\s*', text) if item.strip()]
    if isinstance(parsed, list):
        return [item.strip() for item in parsed if str(item).strip()]
    return [str(parsed).strip()] if str(parsed).strip() else []

In [8]:
df = df_raw.copy()

for column in ['title', 'author', 'summary', 'rating', 'language', 'status', 'url']:
    if column in df.columns:
        df[column] = df[column].map(clean_text)

for column in integer_columns:
    if column in df.columns:
        df[column] = df[column].map(clean_int)

if 'status' in df.columns:
    df['status'] = df['status'].map(clean_status)

if 'last_date' in df.columns:
    df['last_date'] = df['last_date'].map(clean_date)

for column in list_columns:
    if column in df.columns:
        df[column] = df[column].map(parse_list_literal)

for column in ['hits', 'kudos', 'bookmarks']:
    if column in df.columns:
        df[column] = df[column].fillna(0).astype('Int64')

df.head(3)

,title,author,summary,words,current_chapters,total_chapters,kudos,bookmarks,hits,rating,language,status,last_date,warning,category,fandom,relationship,character,freeform,url
0,The man I hate the most 😏.,/users/Pickle_leaf_22/pseuds/Pickle_leaf_22,Aizawa spends time thinking about someone he hates but deep inside he never really hated this so called person. Afte...,1754,1,1,0,0,32,Explicit,English,NaN,NaT,[Creator Chose Not To Use Archive Warnings],[M/M],[僕のヒーローアカデミア | Boku no Hero Academia | My Hero Academia (Anime & Manga)],[Aizawa Shouta | Eraserhead/Yagi Toshinori | All Might],"[Aizawa Shouta | Eraserhead, Yagi Toshinori | All Might]","[Warm and Fuzzy Feelings, Love Confessions, Old man yaoi, Porn with Feelings, Porn With Some Plot, Sex Toys, Lube, C...",https://archiveofourown.org/works/88298181?view_adult=true
1,Glitch,/users/blop_the_bloblin/pseuds/blop_the_bloblin,"Please just read the first chapter if you’re curious, it’s only like, 600 words, and I really don’t know what else t...",642,1,<NA>,6,3,123,Teen And Up Audiences,English,on_going,2026-07-10,[Creator Chose Not To Use Archive Warnings],"[F/M, Other]",[僕のヒーローアカデミア | Boku no Hero Academia | My Hero Academia (Anime & Manga)],"[Aizawa Shouta | Eraserhead & Midoriya Izuku, Midoriya Izuku/Yaoyorozu Momo]","[Midoriya Izuku, Yaoyorozu Momo, Bakugou Katsuki, Aizawa Shouta | Eraserhead, Midoriya Inko, Yagi Toshinori | All Mi...","[Original Character(s), Midoriya Izuku Has a Quirk, Midoriya Izuku Does Not Have One for All Quirk, Slow Burn, Suici...",https://archiveofourown.org/works/88296531?view_adult=true
2,Keeping a Safe Distance,/users/Frol211/pseuds/Frol211,Hawks is sent undercover into the League of Villains. His mission: earn their trust. But the safe distance between h...,2857,2,13,2,0,44,Explicit,English,on_going,2026-07-10,[Creator Chose Not To Use Archive Warnings],[M/M],[僕のヒーローアカデミア | Boku no Hero Academia | My Hero Academia (Anime & Manga)],[Dabi | Todoroki Touya/Takami Keigo | Hawks],"[Dabi | Todoroki Touya, Takami Keigo | Hawks, Shigaraki Tomura | Shimura Tenko, Kurogiri (My Hero Academia), League ...","[Slow Burn, Enemies to Lovers, Mutual Pining, Undercover, Canon Divergence, Angst, Hurt/Comfort, League of Villains,...",https://archiveofourown.org/works/88295646?view_adult=true


In [9]:
df.dtypes

title                  str
author                 str
summary                str
words                int64
current_chapters     int64
total_chapters      object
kudos                Int64
bookmarks            Int64
hits                 Int64
rating                 str
language               str
status                 str
last_date           object
warning             object
category            object
fandom              object
relationship        object
character           object
freeform            object
url                    str
dtype: object

## Basic profiling

A loader should not only clean values, it should also prove the data is internally consistent before inserts.

In [10]:
profile = pd.DataFrame({
    'null_count': df.isna().sum()
})
profile.sort_values('null_count', ascending=False)

,null_count
total_chapters,3660
status,2112
last_date,2112
summary,170
author,105
title,0
words,0
kudos,0
bookmarks,0
hits,0


In [11]:
df[['words', 'kudos', 'bookmarks', 'hits']].describe()

,words,kudos,bookmarks,hits
count,7.040000e+03,7040.0,7040.0,7040.0
mean,3.552882e+04,451.48125,90.785369,15114.328409
std,1.106839e+05,3252.845233,733.284317,122456.48923
min,0.000000e+00,0.0,0.0,0.0
25%,1.986750e+03,5.0,1.0,87.0
50%,6.016500e+03,20.0,3.0,304.0
75%,2.471750e+04,80.0,16.0,1575.0
max,4.096745e+06,101079.0,24639.0,3817351.0


In [12]:
df['status'].value_counts(dropna=False)

status
on_going     4365
NaN          2112
completed     563
Name: count, dtype: int64

## Normalized table previews

These tables mirror the SQL design: one core fic table and separate value tables for the multi-valued tags.

In [13]:
def build_lookup_and_join_tables(frame, column, id_column, value_column):
    # Flatten all values into a single list
    all_values = []

    for values in frame[column]:
        if values:
            all_values.extend(values)

    # Create unique sorted value table
    unique_values = sorted(set(all_values))

    value_table = pd.DataFrame({
        id_column: range(1, len(unique_values) + 1),
        value_column: unique_values
    })

    # Create lookup dictionary
    value_lookup = dict(
        zip(
            value_table[value_column],
            value_table[id_column]
        )
    )

    # Build join table
    join_rows = []

    for fic_id, values in frame[['fic_id', column]].itertuples(index=False):
        if not values:
            continue

        for value in values:
            join_rows.append({
                'fic_id': fic_id,
                id_column: value_lookup[value]
            })

    join_table = (
        pd.DataFrame(join_rows)
        .drop_duplicates()
        .sort_values(['fic_id', id_column])
        .reset_index(drop=True)
    )

    return value_table, join_table

In [14]:
df['fic_id'] = (
    df['url']
    .str.extract(r'/works/(\d+)')
    .astype('Int64')
)

duplicates_removed = (
    df['fic_id']
    .duplicated()
    .sum()
)

df = df.drop_duplicates(
    subset='fic_id',
    keep='first'
).reset_index(drop=True)

df_core = df[
    [
        'fic_id',
        'url',
        'title',
        'author',
        'summary',
        'hits',
        'bookmarks',
        'kudos',
        'current_chapters',
        'total_chapters',
        'words',
        'last_date',
        'status',
        'language',
        'rating'
    ]
].copy()

In [15]:
value_tables = {}
join_tables = {}

table_config = [
    ('warning', 'warning_id', 'warning'),
    ('category', 'category_id', 'category'),
    ('fandom', 'fandom_id', 'fandom'),
    ('relationship', 'relationship_id', 'relationship'),
    ('character', 'character_id', 'character'),
    ('freeform', 'freeform_id', 'freeform')
]

for column, id_column, value_column in table_config:
    if column in df.columns:
        value_table, join_table = build_lookup_and_join_tables(
            df[['fic_id', column]].copy(),
            column,
            id_column,
            value_column
        )

        value_tables[column] = value_table
        join_tables[column] = join_table

## Validation checks

A few simple checks that should pass before the data is trusted for loading.

In [17]:
assert df_core['fic_id'].notna().all()
assert df_core['fic_id'].is_unique
assert df_core[['url', 'title', 'hits', 'bookmarks', 'kudos', 'current_chapters', 'words', 'language', 'rating']].notna().all().all()
for column in list_columns:
    if column in df.columns:
        assert df[column].map(lambda value: isinstance(value, list)).all()
for table in join_tables.values():
    assert table['fic_id'].isin(df_core['fic_id']).all()
for table_name, table in join_tables.items():
    assert not table.empty
for table in join_tables.values():
    assert table.drop_duplicates().shape[0] == table.shape[0]
for name, table in value_tables.items():
    id_column = [c for c in table.columns if c.endswith('_id')][0]
    assert table[id_column].is_unique
'validation passed'

'validation passed'